# MG-GPT under the microscope

Load the MG-GPT checkpoint into `HookedTransformer` (TransformerLens) so we can:

- Run a prompt and **cache every activation** in every layer.
- Visualise **attention patterns** per head per layer with CircuitsVis.
- Watch the prediction sharpen layer-by-layer with a **logit lens**.

The model is a vanilla nanoGPT-style decoder-only transformer, so TransformerLens's
built-in `convert_nanogpt_weights` handles the state-dict remapping. We just need
to build a `HookedTransformerConfig` matching our `medium` preset (8 layers, 8 heads,
512 d_model, 1024 context, GPT-2 BPE vocab).

In [1]:
import os, sys, torch
from pathlib import Path

# Silence the MPS-may-be-incorrect warning on torch 2.7 — we've checked output looks fine.
os.environ.setdefault('TRANSFORMERLENS_ALLOW_MPS', '1')

# Make the project root importable so `model.config` works from inside notebooks/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from model.config import GPTConfig

DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('repo root:', REPO_ROOT)
print('device:   ', DEVICE)

repo root: /Users/lewisdesforges/Documents/TechDev/mg-llm
device:    mps


## 1. Load the checkpoint

In [2]:
CKPT_PATH = REPO_ROOT / 'checkpoints' / 'ckpt_best.pt'
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
cfg: GPTConfig = ckpt['config']
state_dict = ckpt['model']

# Two adjustments to get the real unique-parameter count:
#   1. wte and lm_head are weight-tied — the state dict stores both, count once.
#   2. The causal masks are registered buffers, not parameters (8 × 1024² each).
total = sum(v.numel() for v in state_dict.values() if v.dtype.is_floating_point)
tied = state_dict['lm_head.weight'].numel()
mask_count = sum(v.numel() for k, v in state_dict.items() if k.endswith('.attn.mask'))
unique = total - tied - mask_count

print('checkpoint:', CKPT_PATH.name)
print('iter:     ', ckpt.get('iter_num'))
print('val loss: ', round(ckpt.get('best_val_loss'), 4))
print('config:   ', cfg)
print(f'params:    {unique/1e6:.2f}M trainable  ({mask_count/1e6:.2f}M causal-mask buffers excluded, tied wte/lm_head counted once)')

checkpoint: ckpt_best.pt
iter:      3000
val loss:  3.1388
config:    GPTConfig(block_size=1024, vocab_size=50257, n_layer=8, n_head=8, n_embd=512, dropout=0.1, bias=True)
params:    51.48M trainable  (8.39M causal-mask buffers excluded, tied wte/lm_head counted once)


## 2. Build a matching `HookedTransformerConfig`

TransformerLens needs to know every architectural detail explicitly. The mapping from `GPTConfig` is one-to-one:

| MG-GPT | TransformerLens |
|---|---|
| `n_layer` | `n_layers` |
| `n_head` | `n_heads` |
| `n_embd` | `d_model` |
| `n_embd // n_head` | `d_head` |
| `4 * n_embd` | `d_mlp` |
| `block_size` | `n_ctx` |
| `vocab_size` | `d_vocab` |
| `nn.GELU()` | `act_fn='gelu'` |
| pre-norm LayerNorm with bias | `normalization_type='LN'` |

In [3]:
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.pretrained.weight_conversions import convert_nanogpt_weights

ht_cfg = HookedTransformerConfig(
    n_layers=cfg.n_layer,
    n_heads=cfg.n_head,
    d_model=cfg.n_embd,
    d_head=cfg.n_embd // cfg.n_head,
    d_mlp=4 * cfg.n_embd,
    n_ctx=cfg.block_size,
    d_vocab=cfg.vocab_size,
    act_fn='gelu',
    normalization_type='LN',
    attn_only=False,
    positional_embedding_type='standard',
    tokenizer_name='gpt2',
    device=DEVICE,
)
ht_cfg

HookedTransformerConfig:
{'NTK_by_parts_factor': 8.0,
 'NTK_by_parts_high_freq_factor': 4.0,
 'NTK_by_parts_low_freq_factor': 1.0,
 'NTK_original_ctx_len': 8192,
 'act_fn': 'gelu',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': np.float64(8.0),
 'attn_scores_soft_cap': -1.0,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': None,
 'checkpoint_value': None,
 'd_head': 64,
 'd_mlp': 2048,
 'd_model': 512,
 'd_vocab': 50257,
 'd_vocab_out': 50257,
 'decoder_start_token_id': None,
 'default_prepend_bos': True,
 'device': 'mps',
 'dtype': torch.float32,
 'eps': 1e-05,
 'experts_per_token': None,
 'final_rms': False,
 'from_checkpoint': False,
 'gated_mlp': False,
 'init_mode': 'gpt2',
 'init_weights': True,
 'initializer_range': np.float64(0.035355339059327376),
 'layer_norm_folding': False,
 'load_in_4bit': False,
 'model_name': 'custom',
 'n_ctx': 1024,
 'n_devices': 1,
 'n_heads': 8,
 'n_key_value_heads': None,
 'n_layers': 8,
 'n_params': 2516582

## 3. Convert the state dict and instantiate the model

`convert_nanogpt_weights` does the tensor surgery: splits the combined `c_attn`
matrix into separate Q, K, V per-head matrices, transposes the MLPs, copies
everything across to TransformerLens's naming. We then `load_state_dict` it in.

In [4]:
converted = convert_nanogpt_weights(state_dict, ht_cfg)

# tiktoken-style GPT-2 tokenizer is loaded automatically from HuggingFace.
model = HookedTransformer(ht_cfg)
missing, unexpected = model.load_state_dict(converted, strict=False)
print('missing keys:   ', missing)
print('unexpected keys:', unexpected)
model = model.to(DEVICE).eval()
print('model loaded.')

missing keys:    ['unembed.b_U']
unexpected keys: []
Moving model to device:  mps
model loaded.


## 4. Sanity check: does it generate?

If the weight conversion is correct, this should produce coherent MG-flavoured
text. If the converter is broken, we'd get garbage.

In [5]:
PROMPT = (
    'USER: My 1975 B GT runs rough at idle when warm.\n'
    'ASSISTANT:'
)

out = model.generate(
    PROMPT,
    max_new_tokens=80,
    temperature=0.6,
    top_k=40,
    top_p=0.9,
    return_type='str',
    verbose=False,
)
print(out)

USER: My 1975 B GT runs rough at idle when warm.
ASSISTANT: I’ve had the car rebuilt and replaced all the points, points, points, rotor arm, rotor arm and the HT leads, and coil. The distributor cap was replaced when I installed it, rotor arm last replaced it with a new condenser, rotor arm assembly. The engine was replaced but still new condenser, and the car was still new when it started. I replaced it with


## 5. Run a prompt with the activation cache

`run_with_cache` runs the forward pass and stashes every intermediate tensor
(attention patterns, residual stream at each layer, MLP outputs, etc.) keyed
by hook name. The cache is what every visualisation below reads from.

In [6]:
PROBE = (
    'USER: My MGB GT won\'t start when hot but fine cold.\n'
    'ASSISTANT:'
)

tokens = model.to_tokens(PROBE)
str_tokens = model.to_str_tokens(PROBE)
print('tokens:', tokens.shape)
print('str tokens:', str_tokens)

with torch.no_grad():
    logits, cache = model.run_with_cache(tokens)
print('logits:', logits.shape, ' (batch, seq, vocab)')
print('cache size:', len(cache), 'hook tensors')
print('a few hook names:', list(cache.keys())[:6], '...')

tokens: torch.Size([1, 21])
str tokens: ['<|endoftext|>', 'USER', ':', ' My', ' M', 'GB', ' GT', ' won', "'t", ' start', ' when', ' hot', ' but', ' fine', ' cold', '.', '\n', 'ASS', 'IST', 'ANT', ':']
logits: torch.Size([1, 21, 50257])  (batch, seq, vocab)
cache size: 142 hook tensors
a few hook names: ['hook_embed', 'hook_pos_embed', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.attn.hook_q'] ...


## 6. Attention patterns, layer by layer

Each row of these heatmaps is a query token; each column is a key (a token it
*could* attend to). Bright = strong attention. Causal masking means the upper
triangle is always zero.

Patterns to look for:

- **Diagonal** = head only looks at the current token (often a no-op-ish head).
- **Vertical stripes** = head fixates on one specific past token (e.g. always attending to `MGB` or to a punctuation anchor).
- **Off-diagonal bands** = head looks N tokens back consistently (positional / previous-token head).
- **Sharp diagonals on punctuation** = head boundary-detects.

In [7]:
import circuitsvis as cv
from circuitsvis.attention import attention_patterns

LAYER = 0  # change me — 0..7
attn = cache['pattern', LAYER]  # (batch, head, seq, seq)

# Per-head heatmaps you can browse with the dropdown.
attention_patterns(
    tokens=str_tokens,
    attention=attn[0],  # drop batch dim
)

### All layers at once (multi-head, per layer)

In [8]:
from circuitsvis.attention import attention_heads

# Pick one layer to render a compact grid of all its heads.
LAYER = 4
attention_heads(
    tokens=str_tokens,
    attention=cache['pattern', LAYER][0],
)

## 7. Logit lens — what does each layer 'think' the next token is?

After every transformer block, we apply the final layer norm and the unembedding
matrix to the residual stream. That gives us a per-layer guess of the next token.
Early layers tend to predict generic / high-frequency tokens; later layers sharpen
onto the specific answer.

(This is Nostalgebraist's *logit lens* trick, ~2020.)

In [9]:
import torch.nn.functional as F

POSITION = -1  # last token of the prompt; what comes next?
TOP_K = 5

for layer in range(ht_cfg.n_layers):
    resid = cache['resid_post', layer]            # (1, seq, d_model)
    normed = model.ln_final(resid)                # apply the final LN
    logits_layer = model.unembed(normed)          # project to vocab
    probs = F.softmax(logits_layer[0, POSITION], dim=-1)
    top = torch.topk(probs, TOP_K)
    tokens_top = [model.to_string(t) for t in top.indices]
    print(f'layer {layer}:  ' + '  '.join(f'{repr(t):>14}={p:.3f}' for t, p in zip(tokens_top, top.values.tolist())))

layer 0:            ' I'=0.385          ' The'=0.267           ' My'=0.047           ' if'=0.025             ':'=0.021
layer 1:            ' I'=0.678          ' The'=0.134          ' the'=0.045           ' to'=0.016          ' and'=0.015
layer 2:            ' I'=0.754          ' the'=0.062          ' The'=0.059            ' a'=0.011          ' for'=0.009
layer 3:            ' I'=0.767          ' the'=0.081          ' The'=0.041            ' a'=0.012           ' on'=0.009
layer 4:            ' I'=0.791          ' The'=0.071          ' the'=0.051            ' a'=0.012           ' My'=0.011
layer 5:            ' I'=0.787          ' The'=0.092          ' the'=0.030           ' My'=0.017            ' a'=0.012
layer 6:            ' I'=0.684          ' The'=0.184           ' My'=0.028            ' A'=0.007            ' a'=0.006
layer 7:          ' The'=0.323            ' I'=0.272           ' My'=0.098         ' This'=0.029           ' It'=0.023


## 8. Where to go next

Some directions worth a play:

- **Find a 'previous-token head'**: render attention patterns for all 64 heads, look for ones with a strong off-diagonal one-back band.
- **Persona-vs-no-persona ablation**: cache activations for the same prompt with and without persona priming. Diff the residual stream at each layer — which layer is where the persona 'lives'?
- **Activation patching**: hook into one layer's residual stream, swap it with the activation from a different prompt, see how the output changes. `model.add_hook(name, fn)` is the entry point.
- **Attention head specialisation**: zero-ablate one head at a time during generation and see which heads cause the biggest drop in coherence — the indispensable heads are the interesting ones.

TransformerLens docs: https://transformerlensorg.github.io/TransformerLens/
CircuitsVis: https://github.com/TransformerLensOrg/CircuitsVis